# S1 · UCI HAR 행동 인식 실험

학습 신호의 3축을 함께 회전한 사본을 추가해 착용 각도 변화에 대한 강건성을 확인합니다.

**실행 상태:** 드라이브 원본의 실험 코드 셀에는 저장된 실행 출력이 없습니다. 아래 결과는 실행한 것으로 간주하지 않습니다. 순서대로 실행하고 검증 지표를 기록하세요.

**데이터:** `DATA_PATH`를 현재 Colab의 `DATA/UCI-HAR Dataset` 위치에 맞춰야 합니다. 공식 test를 이용한 반복 모델 선택은 피하세요.



# 실험 S1: 회전 증강 (검증셋으로 판단, 테스트는 마지막에 한 번)

S0 과 모델·학습·평가는 모두 같고, **학습 데이터에 최대 ±15° 무작위 3D 회전 사본을 1벌 추가**한 것만 다릅니다.

| 셀 | 내용 | 언제 실행? |
|---|---|---|
| **셀 1** | 학습 + **검증셋** 평가 (SIT/STAND 핵심 지표 포함) | SEED 0, 1, 2 로 3번 |
| **셀 2** | **최종 테스트** (잠금: `FINAL_TEST=False`) | S0~S3 비교가 끝나고 최종 모델을 정한 뒤 **한 번만** |

> ⚠️ "모두 실행"을 누르면 셀 2는 잠금 메시지(AssertionError)를 내고 멈춥니다. 정상 동작입니다.

## 0. 라이브러리와 실험 설정

필요한 패키지를 불러오고 이 실험의 변경점과 실행 방법을 코드 주석에서 확인합니다.



In [ ]:
# ============================================================
# 실험 S1: 회전 증강 (S0 + 학습 데이터 3D 회전 사본)
# ------------------------------------------------------------
# 기반: CNN_UCI_HAR_S0.ipynb (모델 구조·학습 설정·검증 평가·셀 2 잠금 모두 동일)
#       검증셋으로 판단하고 테스트는 마지막에 한 번만 보는 이유는 S0 헤더 참고
#
# 가설: 모델이 중력 방향(휴대폰 기울기)에만 기대면, 앉음↔섬 기울기 차이가 작은 사람은
#       SIT/STAND 를 구분하기 어렵다. 학습 데이터에 작은 각도의 무작위 회전 사본을 추가하면
#       기울기 단서만으로는 맞히기 어려워져 모델이 움직임 단서를 함께 배우고,
#       기울기 차이가 작은 검증 피험자(5번 12.1°, 17번 14.9°)의 SIT/STAND 정확도가 오른다
#
# 판단 지표 (S0 과 동일)
#   핵심: 검증 피험자 5번, 17번의 SIT/STAND 정확도   → S0 보다 올라야 함
#   보조: 검증 피험자 11, 23, 29번의 SIT/STAND 정확도 → 크게 떨어지면 안 됨
#         (회전이 기울기 차이가 보통인 사람의 중력 단서까지 흐리게 할 수 있음)
#
# 변경 요약 (Ctrl+F 로 "[S1 변경]" 검색)
#   1) 4-2 추가: 학습 윈도우마다 최대 ±15° 무작위 3D 회전 사본 1개를 만들어 이어 붙임 (5650 → 11300)
#   2) 5 정규화: 평균/표준편차는 증강 전 원본 학습 데이터로 계산 (S0 과 같은 기준)
#   3) SUMMARY·그래프·TEST 줄 라벨을 S1 로 변경
#   ※ 검증셋·테스트셋은 회전하지 않음
#
# 실행 방법: 셀 1 만 SEED 0 → 1 → 2 로 바꿔 3번 실행하고 SUMMARY 기록. 셀 2 는 아직 실행하지 않음
#           (학습 데이터가 2배라 에폭당 학습 시간이 S0 의 약 2배)
# ============================================================

import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras.layers import Input, Conv1D, MaxPooling1D, Flatten, Dense
from tensorflow.keras.layers import BatchNormalization, Dropout   # [E1a 변경] 규제용 층
from tensorflow.keras.models import Model
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping             # [E1a 변경] 조기 종료

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)




## 1. Seed

난수 시드를 고정합니다. 시드를 바꿔 재실행하면 초기화에 따른 변동을 볼 수 있습니다.



In [ ]:
# ============================================================
# 1. Seed
# ============================================================

# [E0 변경] 0, 1, 2 로 바꿔 3번 실행 (한 번의 결과는 우연일 수 있음)
SEED = 0

np.random.seed(SEED)
tf.random.set_seed(SEED)




## 2. Data path

Colab에서 Drive를 연결하고 UCI HAR 데이터의 위치를 확인합니다.



In [ ]:
# ============================================================
# 2. Data path
# ============================================================

# [E0 변경] Drive 마운트 + 프로젝트 데이터 경로
from google.colab import drive
drive.mount("/content/drive")

DATA_PATH = "/content/drive/MyDrive/가천대학교/2026-2/1. 기계학습프로그래밍/2week/DATA/UCI-HAR Dataset"

assert os.path.exists(DATA_PATH), f"데이터 경로를 확인하세요: {DATA_PATH}"




## 3. Signal names

입력으로 사용할 가속도계·자이로스코프 9개 채널의 파일명을 정의합니다.



In [ ]:
# ============================================================
# 3. Signal names
# ============================================================

signals = [
    "body_acc_x",
    "body_acc_y",
    "body_acc_z",
    "body_gyro_x",
    "body_gyro_y",
    "body_gyro_z",
    "total_acc_x",
    "total_acc_y",
    "total_acc_z"
]




## 4. Load dataset

공식 train/test 분할의 센서 창, 행동 라벨, 피험자 번호를 읽습니다.



In [ ]:
# ============================================================
# 4. Load dataset
# ============================================================

def load_data(split):

    X = []

    for signal in signals:

        file_path = (
            f"{DATA_PATH}/{split}/"
            f"Inertial Signals/{signal}_{split}.txt"
        )

        data = np.loadtxt(file_path)

        X.append(data)

    # (9, N, 128) -> (N, 128, 9)
    X = np.stack(X, axis=-1)

    y_path = f"{DATA_PATH}/{split}/y_{split}.txt"

    y = np.loadtxt(y_path).astype(int)

    # label 1~6 -> 0~5
    y = y - 1

    # [E0 변경] 피험자 번호(1~30) 로드
    subject_path = f"{DATA_PATH}/{split}/subject_{split}.txt"

    subject = np.loadtxt(subject_path).astype(int)

    return X, y, subject


X_train, y_train, s_train = load_data("train")
X_test, y_test, s_test = load_data("test")




## 4-1. Subject-wise validation split

피험자 5·11·17·23·29를 검증용으로 떼어 학습 피험자와 겹치지 않게 합니다.



In [ ]:
# ============================================================
# 4-1. Subject-wise validation split
# ============================================================

# [E0 변경] 피험자 5명을 통째로 검증셋으로 분리 (같은 사람이 학습·검증에 걸치지 않게)
VAL_SUBJECTS = [5, 11, 17, 23, 29]

val_mask = np.isin(s_train, VAL_SUBJECTS)

X_val, y_val = X_train[val_mask], y_train[val_mask]

# [S0 변경] 검증 윈도우의 피험자 번호 보관
# 이유: 검증 피험자별(특히 5번, 17번) 정확도를 계산하기 위함
s_val = s_train[val_mask]

X_train, y_train = X_train[~val_mask], y_train[~val_mask]


print("\nDataset shape")
print("X_train:", X_train.shape, "(16 subjects)")
print("X_val  :", X_val.shape, "subjects:", VAL_SUBJECTS)
print("X_test :", X_test.shape, "(9 subjects, 셀 2 에서만 사용)")




## 4-2. Rotation augmentation   [S1 추가]

원본 학습 창과 동일 라벨의 ±15° 3D 회전 사본을 만듭니다. 센서 세 묶음에 같은 회전을 적용합니다.



In [ ]:
# ============================================================
# 4-2. Rotation augmentation   [S1 추가]
# ============================================================

# [S1 변경] 학습 윈도우마다 작은 각도의 무작위 3D 회전 사본을 1개씩 만들어 이어 붙임
# 이유: 허리에 찬 휴대폰의 기울기(중력 방향)는 사람마다 다름.
#       모델이 중력 방향에만 기대면 앉음↔섬 기울기 차이가 작은 사람(검증 5번 12.1°, 17번 14.9°)을
#       구분하지 못함 → 착용 각도가 조금씩 다른 데이터를 보여줘서 움직임 단서도 함께 배우게 함

# [S1 변경] 최대 회전 각도 ±15°
# 이유: 학습 피험자의 평균 앉음↔섬 기울기 차이(21.9°)보다 작게 해서 SIT/STAND 단서를 완전히 지우지는 않음.
#       너무 크면(예: ±30°) 앉음과 섬의 기울기 차이 자체가 가려져 오히려 구분이 어려워질 수 있음
#       → 이후 10° / 30° 로 바꿔 비교할 수 있음
MAX_ANGLE_DEG = 15

# [S1 변경] 증강 전 원본 학습 데이터 보관
# 이유: 5번 정규화에서 평균/표준편차를 원본으로 계산해 S0 과 같은 기준으로 비교하기 위함
X_train_orig = X_train

n_windows = X_train.shape[0]

# 1) 윈도우마다 무작위 회전축 (길이 1 인 3차원 단위 벡터)
#    np.random 을 쓰므로 섹션 1 의 SEED 로 재현됨
rot_axis = np.random.normal(size=(n_windows, 3))
rot_axis = rot_axis / np.linalg.norm(rot_axis, axis=1, keepdims=True)

# 2) 윈도우마다 회전 각도: -15° ~ +15° 균등 분포 → 라디안으로 변환
rot_angle = np.deg2rad(np.random.uniform(-MAX_ANGLE_DEG, MAX_ANGLE_DEG, size=n_windows))

# 3) 로드리게스 공식으로 3×3 회전 행렬 만들기: R = I + sin(θ)·K + (1 − cos(θ))·K²
#    K 는 회전축 (kx, ky, kz) 로 만든 반대칭 행렬
kx, ky, kz = rot_axis[:, 0], rot_axis[:, 1], rot_axis[:, 2]
zero = np.zeros(n_windows)

K = np.stack([
    np.stack([zero, -kz, ky], axis=1),
    np.stack([kz, zero, -kx], axis=1),
    np.stack([-ky, kx, zero], axis=1)
], axis=1)                                                  # (N, 3, 3)

sin_t = np.sin(rot_angle)[:, None, None]
cos_t = np.cos(rot_angle)[:, None, None]

R = np.eye(3)[None] + sin_t * K + (1 - cos_t) * (K @ K)     # (N, 3, 3)

# 4) [S1 변경] 같은 회전 행렬을 세 센서(body_acc 0:3, body_gyro 3:6, total_acc 6:9)에 모두 적용
# 이유: 휴대폰은 하나의 강체라 착용 각도가 바뀌면 가속도계와 자이로스코프가 함께 회전함
# 이유: 정규화 전의 원시 물리 단위(g, rad/s)에서 회전해야 실제 착용 각도 변화와 같아짐
X_rot = np.empty_like(X_train)

for start in [0, 3, 6]:

    # 각 윈도우 n, 각 시점 t 의 3축 벡터 v 에 대해 R·v 계산 → (N, 128, 3)
    X_rot[:, :, start:start + 3] = np.einsum(
        "nij,ntj->nti",
        R,
        X_train[:, :, start:start + 3]
    )

# 5) [S1 변경] 원본 뒤에 회전 사본을 이어 붙임 (라벨은 원본과 같음)
# 이유: 검증셋·테스트셋은 회전하지 않음. 실제 측정 데이터 그대로 평가해야 하기 때문
X_train = np.concatenate([X_train, X_rot], axis=0)
y_train = np.concatenate([y_train, y_train], axis=0)

print(
    "Augmented X_train:", X_train.shape,
    f"(원본 {n_windows} + 회전 사본 {n_windows}, 최대 ±{MAX_ANGLE_DEG}°)"
)




## 5. Normalization

학습 데이터의 평균·표준편차만 사용해 검증·테스트 입력을 변환합니다.



In [ ]:
# ============================================================
# 5. Normalization
# ============================================================

# [E0 변경] 평균/표준편차는 학습 16명으로만 계산
# [S1 변경] X_train → X_train_orig (증강 전 원본)으로 통계 계산
# 이유: 회전 사본이 섞이면 채널별 평균·표준편차가 조금 달라짐.
#       S0 과 같은 정규화 기준을 유지해야 "회전 증강"의 효과만 비교할 수 있음
mean = X_train_orig.mean(axis=(0, 1), keepdims=True)
std = X_train_orig.std(axis=(0, 1), keepdims=True)

X_train = (X_train - mean) / (std + 1e-8)
X_val = (X_val - mean) / (std + 1e-8)
X_test = (X_test - mean) / (std + 1e-8)




## 6. One-hot encoding

정수 행동 라벨을 6개 범주의 원핫 벡터로 바꿉니다.



In [ ]:
# ============================================================
# 6. One-hot encoding
# ============================================================

n_outputs = 6

y_train_onehot = to_categorical(
    y_train,
    num_classes=n_outputs
)

y_val_onehot = to_categorical(
    y_val,
    num_classes=n_outputs
)




## 7. CNN Model (E1a 와 동일)

이 실험의 CNN 또는 CNN+LSTM 구조를 정의합니다. 비교 대상과 달라진 층은 코드 주석에 표시했습니다.



In [ ]:
# ============================================================
# 7. CNN Model (E1a 와 동일)
# ============================================================

inputs = Input(shape=(128, 9))


conv1 = Conv1D(
    filters=32,
    kernel_size=3,
    activation="relu"
)(inputs)

conv1 = BatchNormalization()(conv1)   # [E1a 변경] 학습 안정화

pool1 = MaxPooling1D(
    pool_size=2
)(conv1)


conv2 = Conv1D(
    filters=64,
    kernel_size=3,
    activation="relu"
)(pool1)

conv2 = BatchNormalization()(conv2)   # [E1a 변경] 학습 안정화

pool2 = MaxPooling1D(
    pool_size=2
)(conv2)


flat = Flatten()(pool2)

flat = Dropout(0.5)(flat)   # [E1a 변경] 과적합 억제


dense = Dense(
    128,
    activation="relu"
)(flat)

dense = Dropout(0.5)(dense)   # [E1a 변경] 과적합 억제


outputs = Dense(
    n_outputs,
    activation="softmax"
)(dense)




## 8. Model

모델을 만들고 손실 함수, 옵티마이저, 평가 지표를 설정합니다.



In [ ]:
# ============================================================
# 8. Model
# ============================================================

model = Model(
    inputs=inputs,
    outputs=outputs
)


model.compile(
    loss="categorical_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)


model.summary()




## 9. Train

학습 피험자의 데이터로 모델을 훈련하고 학습 이력을 저장합니다.



In [ ]:
# ============================================================
# 9. Train
# ============================================================

# [E1a 변경] val_loss 가 10 에폭 연속 나아지지 않으면 멈추고 최적 가중치로 복원
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model.fit(
    X_train,
    y_train_onehot,
    epochs=100,
    batch_size=64,
    validation_data=(X_val, y_val_onehot),
    shuffle=True,
    callbacks=[early_stop],
    verbose=1
)




## 10. Validation evaluation   [S0 추가]

검증 피험자에 예측을 적용해 정확도·Macro F1과 혼동 행렬을 계산합니다.



In [ ]:
# ============================================================
# 10. Validation evaluation   [S0 추가]
# ============================================================

# [S0 변경] test 대신 검증셋으로 예측하고 평가
# 이유: 방법 비교·선택은 검증셋으로만 함. test 는 셀 2 에서 마지막에 한 번만
y_val_pred = np.argmax(
    model.predict(X_val, verbose=0),
    axis=1
)

class_names = [
    "WALKING",
    "WALKING_UPSTAIRS",
    "WALKING_DOWNSTAIRS",
    "SITTING",
    "STANDING",
    "LAYING"
]

val_accuracy = accuracy_score(y_val, y_val_pred)

val_f1 = f1_score(y_val, y_val_pred, average="macro", zero_division=0)

cm_val = confusion_matrix(y_val, y_val_pred, labels=range(n_outputs))


print("\n==============================")
print("VALIDATION RESULTS (subjects 5, 11, 17, 23, 29)")
print("==============================")

print(f"Accuracy  : {val_accuracy:.4f}")
print(f"F1-score  : {val_f1:.4f}")

print(
    classification_report(
        y_val,
        y_val_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (validation)")
print(cm_val)




## 11. SIT/STAND focus on validation subjects   [S0 추가]

검증 피험자별 SIT/STAND 정확도를 보고 5·17번 핵심 지표와 나머지 지표를 분리합니다.



In [ ]:
# ============================================================
# 11. SIT/STAND focus on validation subjects   [S0 추가]
# ============================================================

# [S0 변경] 검증 피험자별 전체 정확도 + SIT/STAND 정확도
# SIT/STAND 정확도 = 실제로 앉음·섬인 윈도우 중에서 정확히 맞힌 비율 (6개 클래스 예측 기준)
# 이유: 전체 정확도에는 걷기·눕기가 섞여 있어 SIT/STAND 개선이 묻힘 → 문제 클래스만 따로 측정
SIT, STAND = 3, 4

FOCUS_SUBJECTS = [5, 17]   # 기울기 차이가 작은 검증 피험자 (5번 12.1°, 17번 14.9°)

print("\n==============================")
print("PER-SUBJECT (validation)")
print("==============================")
print("subject | all acc | SIT/STAND acc | SIT->STAND | STAND->SIT")

for s in VAL_SUBJECTS:

    m = (s_val == s)

    ms = m & np.isin(y_val, [SIT, STAND])

    sit_to_stand = np.sum(m & (y_val == SIT) & (y_val_pred == STAND))

    stand_to_sit = np.sum(m & (y_val == STAND) & (y_val_pred == SIT))

    mark = "   <- 핵심" if s in FOCUS_SUBJECTS else ""

    print(
        f"{s:7d} | {accuracy_score(y_val[m], y_val_pred[m]):.4f}  | "
        f"{accuracy_score(y_val[ms], y_val_pred[ms]):.4f}        | "
        f"{sit_to_stand:10d} | {stand_to_sit:10d}{mark}"
    )

# 핵심 지표: 5번 + 17번의 SIT/STAND 윈도우를 합쳐서 계산한 정확도
mf = np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

focus_ss_acc = accuracy_score(y_val[mf], y_val_pred[mf])

# 보조 지표: 나머지 검증 피험자(11, 23, 29)의 SIT/STAND 정확도
mo = ~np.isin(s_val, FOCUS_SUBJECTS) & np.isin(y_val, [SIT, STAND])

other_ss_acc = accuracy_score(y_val[mo], y_val_pred[mo])

val_ss_err = cm_val[SIT, STAND] + cm_val[STAND, SIT]

val_dynamic_err = cm_val[0:3, 0:3].sum() - np.trace(cm_val[0:3, 0:3])

best_epoch = int(np.argmin(history.history["val_loss"])) + 1


# [S0 변경] SUMMARY 를 검증 지표로 변경
# [S1 변경] 라벨 S0 → S1
print("\n==============================")
print("SUMMARY (결과표에 기록)")
print("==============================")

print(
    f"S1 | seed {SEED} | val acc {val_accuracy:.4f} | val F1 {val_f1:.4f} | "
    f"focus SIT/STAND(5,17) {focus_ss_acc:.4f} | other SIT/STAND(11,23,29) {other_ss_acc:.4f} | "
    f"val SIT<->STAND {val_ss_err} ({cm_val[SIT, STAND]}+{cm_val[STAND, SIT]}) | val dynamic {val_dynamic_err} | "
    f"epochs {len(history.history['val_loss'])} (best {best_epoch}) | params {model.count_params():,}"
)




## 12. Learning curve

학습 곡선을 보고 손실과 검증 지표가 에폭에 따라 어떻게 변하는지 살펴봅니다.



In [ ]:
# ============================================================
# 12. Learning curve
# ============================================================

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["loss"], label="train loss")
plt.plot(history.history["val_loss"], label="val loss")
plt.xlabel("epoch")
plt.title(f"S1 loss (seed {SEED})")   # [S1 변경] 라벨
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["accuracy"], label="train acc")
plt.plot(history.history["val_accuracy"], label="val acc")
plt.xlabel("epoch")
plt.title(f"S1 accuracy (seed {SEED})")
plt.legend()

plt.tight_layout()
plt.show()

## 🔒 최종 테스트 (한 번만 실행)
S0~S3 을 **검증 지표(5·17번 SIT/STAND 정확도)**로 비교해 최종 방법을 정한 뒤에만 실행합니다.
실행 순서: 셀 1 을 원하는 SEED 로 실행 → 아래 셀에서 `FINAL_TEST = True` 로 바꿔 실행

In [ ]:
# ============================================================
# 13. FINAL TEST   [S0 추가]
# ============================================================

# [S0 변경] 테스트 평가를 별도 셀로 분리하고 잠금
# 이유: test 점수를 보고 방법을 고르면 test 에 맞춘 선택이 되어 최종 점수의 의미가 사라짐.
#       검증셋으로 최종 방법을 정한 뒤에만 True 로 바꿔 실행 (셀 1 에서 학습한 model 을 그대로 평가)
FINAL_TEST = False

assert FINAL_TEST, (
    "테스트셋은 최종 모델을 정한 뒤 한 번만 봅니다. "
    "지금은 셀 1 의 검증 결과로 비교하세요. (최종 평가 때 FINAL_TEST = True)"
)

y_pred = np.argmax(
    model.predict(X_test, verbose=0),
    axis=1
)

accuracy = accuracy_score(y_test, y_pred)

f1 = f1_score(y_test, y_pred, average="macro", zero_division=0)

cm = confusion_matrix(y_test, y_pred, labels=range(n_outputs))


print("\n==============================")
print("FINAL TEST RESULTS")
print("==============================")

print(f"Accuracy  : {accuracy:.4f}")
print(f"F1-score  : {f1:.4f}")

print(
    classification_report(
        y_test,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

print("CONFUSION MATRIX (test)")
print(cm)


print("\n==============================")
print("PER-SUBJECT ACCURACY (test)")
print("==============================")

subject_acc = {}

for s in np.unique(s_test):

    m = (s_test == s)

    subject_acc[s] = accuracy_score(y_test[m], y_pred[m])

    print(f"subject {s:2d}: {subject_acc[s]:.4f}")

worst_subject = min(subject_acc, key=subject_acc.get)

sit_stand_err = cm[SIT, STAND] + cm[STAND, SIT]

dynamic_err = cm[0:3, 0:3].sum() - np.trace(cm[0:3, 0:3])

print(
    f"\nS1 TEST | seed {SEED} | acc {accuracy:.4f} | macro F1 {f1:.4f} | "
    f"SIT<->STAND {sit_stand_err} | dynamic {dynamic_err} | "
    f"worst subject {worst_subject} ({subject_acc[worst_subject]:.4f})"
)